# Week 18: Autonomous AI Agents & Multi-Agent Systems
## Notebook 07: Autonomous Agent Benchmarking & SWE-bench Evaluation from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Early AI benchmarks (MMLU, HumanEval) tested simple multiple-choice questions or isolated 10-line Python functions.
However, real-world software engineering is infinitely more complex:
- Cloning a 100,000-line repository (e.g. Django, SymPy, scikit-learn).
- Understanding complex GitHub issues with reproduction steps.
- Editing files across multiple subdirectories.
- Generating a clean **Unified Git Diff Patch** that passes existing and new regression unit tests.

Introduced by Princeton in 2023, **SWE-bench** is the gold standard benchmark evaluating frontier software engineering agents (Devin, SWE-agent, GitHub Copilot Workspace).

---

### 2. The Evaluation Protocol
For each task instance:
1. Environment is initialized in a headless Docker container at a specific historical git commit.
2. The agent is provided with the Issue Description and terminal tools.
3. The agent outputs a unified patch: `git diff > patch.diff`.
4. The test harness applies the patch and executes **Fail-to-Pass (F2P)** and **Pass-to-Pass (P2P)** test assertions.
   - **Resolved Status:** All F2P tests pass AND no P2P tests regress!

In [ ]:
import difflib

# Step 1: Ground Truth Repository State
original_code = """def calculate_discount(price, customer_type):
    if customer_type == "VIP":
        return price * 0.20
    return 0.0
"""

# GitHub Issue: "calculate_discount crashes with TypeError when price is negative!"
# Agent generates proposed patch:
patched_code = """def calculate_discount(price, customer_type):
    if price < 0:
        raise ValueError("Price cannot be negative")
    if customer_type == "VIP":
        return price * 0.20
    return 0.0
"""

# Step 2: Generate Unified Git Diff Patch
orig_lines = original_code.splitlines(keepends=True)
patch_lines = patched_code.splitlines(keepends=True)

diff = difflib.unified_diff(
    orig_lines,
    patch_lines,
    fromfile="a/billing/discount.py",
    tofile="b/billing/discount.py"
)
diff_patch = "".join(diff)
print("--- Generated Agent Git Diff Patch ---")
print(diff_patch)

# Step 3: Run Automated Test Harness (Fail-to-Pass & Pass-to-Pass)
class TestHarness:
    def __init__(self, target_func):
        self.func = target_func

    def run_tests(self):
        results = {"P2P": False, "F2P": False}
        
        # 1. Pass-to-Pass (Existing functionality must not break)
        try:
            assert self.func(100, "VIP") == 20.0
            assert self.func(100, "Regular") == 0.0
            results["P2P"] = True
        except Exception:
            results["P2P"] = False
            
        # 2. Fail-to-Pass (The bug reported in GitHub issue must be resolved)
        try:
            try:
                self.func(-50, "VIP")
                results["F2P"] = False # Should have raised ValueError!
            except ValueError:
                results["F2P"] = True
        except Exception:
            results["F2P"] = False
            
        return results

# Compile patched function
exec_namespace = {}
exec(patched_code, exec_namespace)
patched_func = exec_namespace["calculate_discount"]

harness = TestHarness(patched_func)
eval_results = harness.run_tests()

print("\n--- SWE-bench Evaluation Results ---")
print(f"Pass-to-Pass (No Regressions):    {eval_results['P2P']}")
print(f"Fail-to-Pass (Bug Resolved):       {eval_results['F2P']}")
resolved = eval_results['P2P'] and eval_results['F2P']
print(f"Overall Task Status:              {'RESOLVED ✅' if resolved else 'FAILED ❌'}")

### 3. Step-by-Step Code Walkthrough
1. **Diff Generation:** Generates standard POSIX unified patches identical to `git diff`.
2. **Dual-Criteria Assertion:** Verifies that existing functionality remained intact (Pass-to-Pass) while confirming the reported issue was fixed (Fail-to-Pass).

---

### 4. Industry Applications & Trade-offs
- **Autonomous Bug Fixing:** Automated nightly resolution of low-priority customer-reported GitHub issues.
- **Pass@1 Tracking:** State-of-the-art agents currently solve ~40-50% of real-world SWE-bench Verified tasks.

---

### 5. Top Interview Questions & Answers
**Q1: What is the difference between Fail-to-Pass (F2P) and Pass-to-Pass (P2P) tests in SWE-bench?**
> **Answer:** F2P tests are unit tests that failed on the unpatched codebase (capturing the specific reported bug). P2P tests are all existing unit tests that were already passing. A patch is only valid if it turns F2P tests green without breaking any P2P tests (zero regressions).

**Q2: Why is testing autonomous agents in isolated containers mandatory?**
> **Answer:** Autonomous agents have terminal execution privileges. An agent trying to resolve a dependency issue might run `pip install`, alter system libraries, or format disks. Ephemeral Docker containers ensure reproducible state and total host security.